In [12]:
import os, pandas as pd
from huggingface_hub import login, list_repo_files, hf_hub_download
login('')  

files = list_repo_files("BibbyResearch/ProfanityBench", repo_type="dataset")
print(files)


['.gitattributes', 'README.md', 'dataset.jsonl', 'plots/category_cooccurrence.png', 'plots/category_distribution.png', 'plots/category_language_heatmap.png', 'plots/category_tone_heatmap.png', 'plots/common_literal_translations.png', 'plots/content_flags.png', 'plots/correlation_matrix.png', 'plots/dashboard_overview.png', 'plots/dashboard_sociolinguistic.png', 'plots/english_loanwords.png', 'plots/entries_per_country.png', 'plots/entries_per_language.png', 'plots/etymology_length.png', 'plots/etymology_sources.png', 'plots/flag_cooccurrence.png', 'plots/generation_analysis.png', 'plots/generation_category_heatmap.png', 'plots/genz_categories.png', 'plots/hate_speech_analysis.png', 'plots/language_script_heatmap.png', 'plots/languages_per_country.png', 'plots/missing_values.png', 'plots/popularity_distribution.png', 'plots/popularity_vs_severity.png', 'plots/script_distribution.png', 'plots/severity_boxplot_by_category.png', 'plots/severity_by_category.png', 'plots/severity_by_generati

In [13]:
import os, pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download("BibbyResearch/ProfanityBench",
                       filename="dataset.jsonl", repo_type="dataset")
df = pd.read_json(path, lines=True)
print(df.shape)

C:\Users\Artyom\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Artyom\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dataset.jsonl: reconstructing file:   0%|          |  0.00B / 68.1MB            

dataset.jsonl: downloading bytes:           |  0.00B            

C:\Users\Artyom\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Artyom\.cache\huggingface\hub\datasets--BibbyResearch--ProfanityBench. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


(82208, 24)


In [14]:
def has(col, pat):
    return df[col].astype(str).str.contains(pat, case=False, na=False)

mask = (has("language", r"kazakh|^kk$|qazaq") |
        has("country", r"kazakh") |
        has("region", r"kazakh") |
        has("notes", r"kazakh"))
print("Kazakh-related rows:", mask.sum())
print(df[mask]["language"].value_counts())
print(df[mask][["language","script","word","severity","category","example_sentence"]].head(30).to_string())

Kazakh-related rows: 134
language
Kazakh    132
Nogai       2
Name: count, dtype: int64
      language    script        word  severity      category                                       example_sentence
3456    Kazakh  Cyrillic         шөп         3    drug_slang                                     Шөп шегіп отырмын.
3457    Kazakh  Cyrillic       анаша         3    drug_slang                                  Анаша сататындар көп.
3458    Kazakh  Cyrillic      көкнәр         3    drug_slang                                 Көкнәр алқабы табылды.
3459    Kazakh  Cyrillic    кристалл         3    drug_slang                                    Кристалл қолданады.
3460    Kazakh  Cyrillic       гашиш         3    drug_slang                                         Гашиш табылды.
14080   Kazakh  Cyrillic   Айналайын         1         slang                      Айналайын, сен неге сонша тамаша!
14081   Kazakh  Cyrillic       Кекір         1         slang                                    Сен 

In [15]:
kz = df[df["language"] == "Kazakh"].copy()
print(kz["category"].value_counts())
print(kz["severity"].value_counts().sort_index())
print(kz["script"].value_counts())

# see everything, a few columns at a time
pd.set_option("display.max_rows", 200, "display.width", 200)
print(kz[["word","category","severity","safe_for_work","hate_speech","sexual","actual_meaning"]]
      .sort_values(["category","severity"]).to_string())

category
slang                                28
code-switching slang                 20
regional dialect slang               19
class_insult                         10
misogynistic slur                     8
intelligence insult (gullibility)     8
profanity                             8
slur                                  8
disability                            6
drug_slang                            5
scatological                          5
ethnic_slur                           3
illegitimacy                          3
rural/urban slur                      1
Name: count, dtype: int64
severity
1    48
2    31
3    21
4    13
5    19
Name: count, dtype: int64
script
Cyrillic    132
Name: count, dtype: int64
              word                           category  severity  safe_for_work  hate_speech  sexual                                                                                                 actual_meaning
27930          бай                       class_insult         3       

In [16]:
import re, os
from pathlib import Path

keep_cats = ["profanity","scatter" if False else "scatological","misogynistic slur","slur",
             "ethnic_slur","illegitimacy",
             "intelligence insult (gullibility)","disability","rural/urban slur"]

kz = df[(df["language"] == "Kazakh") & (df["category"].isin(keep_cats))].copy()

def norm(w):
    w = str(w).strip().lower()
    return re.sub(r"\s+", " ", w)

kz["word_norm"] = kz["word"].map(norm)
lex = (kz[["word_norm","word","category","severity","actual_meaning","script"]]
         .drop_duplicates("word_norm")
         .sort_values(["category","word_norm"])
         .reset_index(drop=True))
lex["source"] = "BibbyResearch/ProfanityBench"

Path("data").mkdir(exist_ok=True)
lex.to_csv("data/lexicon_profanitybench_kk.csv", index=False, encoding="utf-8")
print(len(kz), "rows ->", len(lex), "unique words")
print(lex["category"].value_counts())
print(lex["script"].value_counts())

50 rows -> 50 unique words
category
intelligence insult (gullibility)    8
misogynistic slur                    8
profanity                            8
slur                                 8
disability                           6
scatological                         5
ethnic_slur                          3
illegitimacy                         3
rural/urban slur                     1
Name: count, dtype: int64
script
Cyrillic    50
Name: count, dtype: int64


In [17]:
import os, pandas as pd
from huggingface_hub import login, list_repo_files, hf_hub_download

print(list_repo_files("issai/kazsandra", repo_type="dataset"))

full_path = hf_hub_download("issai/kazsandra", filename="full/full.csv", repo_type="dataset")
full = pd.read_csv(full_path)
print(full.shape)
print(full.columns.tolist())
print(full["domain"].value_counts())

market = full[full["domain"].str.lower() == "market"].copy()
print(market.shape)
print(market["label"].value_counts().sort_index())
print(market["text"].str.len().describe())

['.gitattributes', 'README.md', 'full/full.csv', 'polarity_classification/test_pc.csv', 'polarity_classification/train_pc.csv', 'polarity_classification/valid_pc.csv', 'score_classification/test_sc.csv', 'score_classification/train_sc.csv', 'score_classification/valid_sc.csv']


full/full.csv: reconstructing file:   0%|          |  0.00B / 24.4MB            

full/full.csv: downloading bytes:           |  0.00B            

C:\Users\Artyom\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Artyom\.cache\huggingface\hub\datasets--issai--kazsandra. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


(180064, 4)
['custom_id', 'text', 'label', 'domain']
domain
appstore     135073
market        30289
mapping        8897
bookstore      5805
Name: count, dtype: int64
(30289, 4)
label
1     1043
2      350
3      913
4     2775
5    25208
Name: count, dtype: int64
count    30289.000000
mean        66.116973
std         57.825325
min          1.000000
25%         30.000000
50%         49.000000
75%         82.000000
max        900.000000
Name: text, dtype: float64


In [18]:
import re, unicodedata, pandas as pd, numpy as np

lex = pd.read_csv("data/lexicon_profanitybench_kk.csv")

# --- normalization used for matching only (original text is kept) ---
LAT2CYR = str.maketrans({"a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","h":"н","b":"в"})
def norm_text(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = s.replace("ё","е")
    s = re.sub(r"\s+", " ", s).strip()
    return s
def match_form(s):
    # map Latin look-alikes to Cyrillic, so mixed-script spellings still match
    return norm_text(s).translate(LAT2CYR)

# --- 1. dedup (exact, on normalized text) ---
market = market.copy()
market["text_norm"] = market["text"].map(norm_text)
n0 = len(market)
market = market.drop_duplicates("text_norm").reset_index(drop=True)
print(f"dedup: {n0} -> {len(market)}")

# --- 2. flag with lexicon (prefix match on word start; exact match for very short words) ---
def build_pattern(w):
    w = re.escape(match_form(w))
    return rf"(?<![а-яәіңғүұқөһa-z0-9]){w}" + (r"(?![а-яәіңғүұқөһ])" if len(w) <= 3 else r"[а-яәіңғүұқөһ]*")

lex["pat"] = lex["word_norm"].map(build_pattern)
market["match_text"] = market["text"].map(match_form)

def find_hits(t):
    return [r.word_norm for r in lex.itertuples() if re.search(r.pat, t)]
market["hits"] = market["match_text"].map(find_hits)
market["flagged"] = market["hits"].map(len) > 0

print("flagged:", market["flagged"].sum(), "of", len(market))
print(market.groupby("label")["flagged"].agg(["sum","count"]))
print(pd.Series([w for h in market["hits"] for w in h]).value_counts().head(40))

dedup: 30289 -> 29910
flagged: 60 of 29910
       sum  count
label            
1        7   1030
2        3    333
3        3    868
4        8   2613
5       39  25066
дәрет      18
қатын      14
көтен      14
ақымақ      3
мыстан      3
жынды       2
соқыр       1
ит          1
дұшпан      1
мүгедек     1
нәжіс       1
зәр         1
есек        1
Name: count, dtype: int64


In [19]:
full = full.copy()
full["text_norm"] = full["text"].map(norm_text)
full = full.drop_duplicates("text_norm").reset_index(drop=True)
full["match_text"] = full["text"].map(match_form)
full["hits"] = full["match_text"].map(find_hits)
full["flagged"] = full["hits"].map(len) > 0

print(full.groupby("domain")["flagged"].agg(flagged="sum", total="count"))
print((full.groupby("domain")["flagged"].mean()*100).round(3))

           flagged   total
domain                    
appstore      1345  132182
bookstore       32    5710
mapping         89    8659
market          60   29880
domain
appstore     1.018
bookstore    0.560
mapping      1.028
market       0.201
Name: flagged, dtype: float64


In [20]:
cnt = pd.Series([w for h in full["hits"] for w in h]).value_counts()
print(cnt.to_string())

for w in cnt.index[:15]:
    sub = full[full["hits"].map(lambda h: w in h)]
    print(f"\n=== {w} | n={len(sub)} | share of 5★={(sub.label==5).mean():.2f}")
    for t in sub.sample(min(6, len(sub)), random_state=0)["text"]:
        print("  -", t[:140].replace("\n", " "))

қотақ        568
қотақбас     236
жынды        218
ам           124
қатын         93
дәрет         80
топас         77
ит            72
ақымақ        60
ұятсыз        58
көт           46
арам          36
есек          34
көтен         21
мүгедек       17
шошқа         16
бок           13
надан         11
заңсыз        11
соқыр          8
тентек         8
арсыз          6
жәлеп          5
мыстан         4
саңырау        4
қақбас         4
си             4
амшелек        3
жезөкше        3
нәжіс          2
зәр            2
ақсақ          2
жетесіз        2
кеще           2
дұшпан         1
шалақазақ      1
мылқау         1
ақмак          1
аңқау          1
жаляп          1
мақау          1

=== қотақ | n=568 | share of 5★=0.38
  - Қотақбастар бір нәрсе дұрыс жасай алмайды. Қолыңыздан келмейді екен көттеріңді қысып үйде отырсаңдаршы
  - Далбайоп қотақ зайбол
  - Қотақбастар сігілің рекламарыңмен
  - Қотақ бас оян екен
  - Шарейтті керемет жақсы дегендер қотақ жаман нашар дегендер маладес 

In [21]:
RU_MAT = re.compile(
    r"(?<![а-я])[а-я]*(?:хуй|хуе|хуя|хуи|пизд|ебан|ебат|ебал|ебну|заеб|уеб|долбаеб|далбаеб|мудак|мудил|пидор|пидар|говн)"
    r"|(?<![а-я])бля(?![а-я])", re.I)

full["ru_flag"] = full["match_text"].str.contains(RU_MAT)
print(full.groupby("domain")["ru_flag"].agg(flagged="sum", total="count"))
print(pd.crosstab(full["flagged"], full["ru_flag"]))
for t in full[full["ru_flag"]].sample(15, random_state=0)["text"]:
    print("-", t[:140].replace("\n", " "))

           flagged   total
domain                    
appstore       520  132182
bookstore        2    5710
mapping          1    8659
market           0   29880
ru_flag   False  True 
flagged               
False    174451    454
True       1457     69
- Не советую 1 сагатка алдк істими калп кайта кайта 11 мын тг алп калд call центр жок согп причина сурайн десек полный говно алмаң лучше яндек
- ШЕШЕН АМЫ БЛЯ
- Қалай мынау түсінбедім Бырак кушты.автобус каякта кебатканын бле аламын
- Керемет Осы оиын пиздес
- Ассалаумағалейкум каймады қазақ бауырлар! Бұл сайт пирамида секілді, қайсына ақша салсаңыз соны қымбат қылып қояды, бітпейтін аксия, клиент 
- Қотақбастар Югта Автобус нехуя жоқ кім кінәлі Умный Транспорт Кінәлі щщс
- Пидараз очен плахой буква уапше нет Мал тартпандар әріптер жок мен айтарымды айтым оздерын біліндер
- Пиздец ойын айтайын
- Мегабайт пиздец жейді шш 👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎👎
- Ашылмайды ғо бля🤬🤮
- ой пизду казакша рэптер жок паражняк ко
- Бля кірмей 

In [22]:
L = "а-яәіңғүұқөһ"
tierA = ["қотақ","қотақбас","қақбас","бок","амшелек","жезөкше","жәлеп","жаляп"]
tierB = ["көт","көтен","ам","си","қатын","ит","топас","ақымақ","жынды","есек","надан","ұятсыз",
         "арсыз","аңқау","жетесіз","шошқа","мыстан","кеще","шалақазақ","сіг","сік"]
short = {"көт","ам","си","қатын","ит","бок","сіг","сік"}
SUF = r"(?:[ыі]ң(?:ды|ді|ға|ге)?|[ыі]м|[ыі]н|[ыі]|тар|тер|дар|дер|лар|лер|ты|ті|ды|ді|да|де|ға|ге|қа|ке|мен|бен|пен)"

def pat(w):
    m = re.escape(match_form(w))
    tail = rf"(?:{SUF}){{0,3}}(?![{L}])" if w in short else rf"[{L}]*"
    return rf"(?<![{L}a-z0-9]){m}{tail}"
compile_tier = lambda ws: [(w, re.compile(pat(w))) for w in ws]
PA, PB = compile_tier(tierA), compile_tier(tierB)

RU_PRE = r"(?:не|ни|на|по|за|до|от|вы|у|про|раз|пере|при|под)?"
RU_ROOT = r"(?:хуй|хуе|хуя|хуи|пизд|бляд|блять|ебан|ебат|ебал|ебну|ебись|долбаеб|далбаеб|далбайоп|мудак|мудил|пидор|пидар|сука|сучк|говн|гандон)"
RU_MAT = re.compile(rf"(?<![{L}a-z]){RU_PRE}{RU_ROOT}[{L}]*|(?<![{L}])бля(?![{L}])")

hits = lambda t, P: [w for w, p in P if p.search(t)]
full["hitA"]   = full["match_text"].map(lambda t: hits(t, PA))
full["hitB"]   = full["match_text"].map(lambda t: hits(t, PB))
full["ru_hit"] = full["match_text"].map(lambda t: bool(RU_MAT.search(t)))
full["flag_any"] = full.hitA.map(len).gt(0) | full.hitB.map(len).gt(0) | full.ru_hit

print(full.groupby("domain")["flag_any"].agg(flagged="sum", total="count"))
print("A:", (full.hitA.map(len)>0).sum(), "| ru:", full.ru_hit.sum(),
      "| B-only:", (full.hitB.map(len).gt(0) & ~full.hitA.map(len).gt(0) & ~full.ru_hit).sum())

# labeling sheets (kk_prof / ru_prof are filled by you; 'proposed' = 1 for tier A or Russian hits)
cand = full[full.flag_any].copy()
cand["hits"] = (cand.hitA + cand.hitB).map(",".join)
cand["proposed"] = np.where(cand.hitA.map(len).gt(0) | cand.ru_hit, 1, np.nan)
for c in ["kk_prof", "ru_prof", "notes"]: cand[c] = ""
cols = ["custom_id","domain","label","text","hits","ru_hit","proposed","kk_prof","ru_prof","notes"]
cand.sort_values(["domain","proposed"])[cols].to_csv("data/label_flagged.csv", index=False, encoding="utf-8-sig")

unf = full[~full.flag_any]; low = unf.label.isin([1, 2])
take = lambda d, n: d.sample(min(n, len(d)), random_state=42)
smp = pd.concat([take(unf[(unf.domain=="appstore") & low], 300),
                 take(unf[(unf.domain=="market") & low], 150),
                 take(unf[(unf.domain=="market") & ~low], 100),
                 take(unf[unf.domain.isin(["mapping","bookstore"]) & low], 100)])
smp[["custom_id","domain","label","text"]].assign(kk_prof="", ru_prof="", notes="") \
   .to_csv("data/label_unflagged_sample.csv", index=False, encoding="utf-8-sig")
print(len(cand), len(smp))

           flagged   total
domain                    
appstore      1990  132182
bookstore       22    5710
mapping         21    8659
market          43   29880
A: 590 | ru: 624 | B-only: 946
2076 650


In [23]:
import re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

SEED = 42
fl = pd.read_csv("data/label_flagged.csv", encoding="utf-8-sig");          fl["source"] = "flagged"
un = pd.read_csv("data/label_unflagged_sample.csv", encoding="utf-8-sig"); un["source"] = "sample"
pool = pd.concat([fl, un], ignore_index=True)

def gkey(s):
    s = re.sub(r"[^а-яәіңғүұқөһa-z]", "", match_form(s))
    return re.sub(r"(.)\1+", r"\1", s)                    # collapse repeated letters
pool["gkey"] = pool["text"].map(gkey)
txt = pool["gkey"].where(pool["gkey"].str.len() > 0, pool["text"].astype(str))

# near-duplicate groups (used for grouping only, not as model features)
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2).fit_transform(txt)
S = cosine_similarity(X)
_, pool["group"] = connected_components(csr_matrix(S >= 0.85), directed=False)

# group-level split, stratified by source and whether the group has a proposed-1 row
g = pool.groupby("group").agg(src=("source", "first"),
                              prop=("proposed", lambda s: s.notna().any())).reset_index()
g["split"] = ""
rng = np.random.default_rng(SEED)
for _, idx in g.groupby(["src", "prop"]).groups.items():
    idx = rng.permutation(np.array(list(idx)))
    a, b = int(.70 * len(idx)), int(.85 * len(idx))
    g.loc[idx[:a], "split"] = "train"; g.loc[idx[a:b], "split"] = "val"; g.loc[idx[b:], "split"] = "test"
pool["split"] = pool["group"].map(g.set_index("group")["split"])

print(pool.groupby(["split", "source"]).size().unstack())
print(pool[pool.proposed.notna()].groupby("split").size())
print(pool.groupby(["split", "domain"]).size().unstack())

order = {"test": 0, "val": 1, "train": 2}
for name, d in [("flagged", pool[pool.source == "flagged"]), ("unflagged_sample", pool[pool.source == "sample"])]:
    (d.sort_values("split", key=lambda s: s.map(order))
       .drop(columns=["source", "gkey"])
       .to_csv(f"data/label_{name}_split.csv", index=False, encoding="utf-8-sig"))

source  flagged  sample
split                  
test        312      98
train      1449     453
val         315      99
split
test     169
train    788
val      173
dtype: int64
domain  appstore  bookstore  mapping  market
split                                       
test         347         10        8      45
train       1596         46       53     207
val          347          9       17      41


In [24]:
f = pd.read_csv("data/label_flagged_split.csv", encoding="utf-8-sig")
s = pd.read_csv("data/label_unflagged_sample_split.csv", encoding="utf-8-sig")
print("rows:", len(f), len(s))

# 1. typos / invalid values in the label columns
for name, d in [("flagged", f), ("sample", s)]:
    raw = d["kk_prof"].dropna().astype(str).str.strip()
    print(name, "| invalid kk_prof:", raw[~raw.isin(["0", "1", "0.0", "1.0"])].value_counts().to_dict(),
          "| ru_prof filled:", d["ru_prof"].notna().sum())
    d["kk_prof"] = pd.to_numeric(d["kk_prof"], errors="coerce")
    d["ru_prof"] = pd.to_numeric(d["ru_prof"], errors="coerce")

# 2. test split, flagged file
t = f[f.split == "test"].copy()
t["reviewed"] = t.kk_prof.notna() | t.ru_prof.notna()
t["y"] = np.where(t.reviewed, ((t.kk_prof == 1) | (t.ru_prof == 1)).astype(int), t.proposed)
ver = t[t.proposed == 1]
bonly = t[t.proposed.isna()]
print("\nTEST flagged:", len(t), "| verify rows:", len(ver), "| B-only rows:", len(bonly))
print("verify rows kept as 1:", int((ver.y == 1).sum()), "| overridden to 0:", int((ver.y == 0).sum()),
      "| lexicon precision on test:", round((ver.y == 1).mean(), 3))
print("B-only labeled:", int(bonly.reviewed.sum()), "of", len(bonly),
      "| of those, positives:", int((bonly[bonly.reviewed].y == 1).sum()))

# which words produced the overridden (false) hits
ex = ver[ver.y == 0].assign(hit=ver[ver.y == 0].hits.fillna("").str.split(",")).explode("hit")
print("\nwords in overridden rows:\n", ex.hit.value_counts().head(15).to_string())
print("\noverridden examples:")
for r in ver[ver.y == 0].head(12).itertuples(): print(" -", r.hits, "|", str(r.text)[:110].replace("\n", " "))
print("\nrandom accepted examples (skim these):")
for r in ver[ver.y == 1].sample(min(12, int((ver.y == 1).sum())), random_state=1).itertuples():
    print(" -", r.hits, "|", str(r.text)[:110].replace("\n", " "))

# 3. test split, random sample file (blank = assumed 0)
ts = s[s.split == "test"].copy()
ts["y"] = np.where(ts.kk_prof.notna() | ts.ru_prof.notna(), ((ts.kk_prof == 1) | (ts.ru_prof == 1)).astype(int), 0)
print("\nTEST sample:", len(ts), "| positives found (lexicon misses):", int(ts.y.sum()))
print(ts[ts.y == 1][["domain", "text"]].head(10).to_string())

# 4. leakage check across splits (same normalized text in more than one split)
pool = pd.concat([f[["text", "split"]], s[["text", "split"]]])
pool["gkey"] = pool.text.map(gkey)
multi = pool[pool.gkey != ""].groupby("gkey").split.nunique()
print("\nnormalized texts present in >1 split:", int((multi > 1).sum()))

# 5. hardware
try:
    import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)

rows: 2076 650
flagged | invalid kk_prof: {} | ru_prof filled: 0
sample | invalid kk_prof: {} | ru_prof filled: 0

TEST flagged: 312 | verify rows: 169 | B-only rows: 143
verify rows kept as 1: 169 | overridden to 0: 0 | lexicon precision on test: 1.0
B-only labeled: 0 of 143 | of those, positives: 0

words in overridden rows:
 Series([], )

overridden examples:

random accepted examples (skim these):
 - nan | Далбайоп ойын 🖕🖕🖕🖕🖕🖕
 - қотақ,ам | Секс ам,қотақтар мен сендерді сігем қыздар.😎😀❤☎️қазақ барма.
 - қотақ | Мал ойыг қотақ ойын
 - қотақ | Мынаны жасаған адам тошна қотақ бас екен
 - қотақ,қотақбас,жәлеп | Шешең сгиін қотақбас әм жәлеп малдар сет ұстатыңдар өліп қалғыр ешактар
 - nan | Анасыгин пиздес қо мынау
 - қотақ,қотақбас | Қотақбастар вирус кырып кетті телефоныма
 - nan | Пиздец ойын айтайын
 - nan | ГОВНО! АЙТАТЫН СӨЗ ЖОҚ. ОСЫНДАЙ БӘЛЕНІ ШЫҒАРМАЙАҚ ҚОЙМАДЫҢДАРМА. ПОЛНЫЙ \nТУПИЗМ. Шығарған адам ренжіме. Қолың
 - nan | нихуя истемейди регистрация жок,котагым жеген нарсе
 - а

In [26]:
import os, re, hashlib, unicodedata
import numpy as np, pandas as pd
from pathlib import Path
from getpass import getpass
from huggingface_hub import login, hf_hub_download

# --- helpers ---
LAT2CYR = str.maketrans({"a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","h":"н","b":"в"})
def norm_text(s):
    s = unicodedata.normalize("NFKC", str(s)).lower().replace("ё", "е")
    return re.sub(r"\s+", " ", s).strip()
def match_form(s): return norm_text(s).translate(LAT2CYR)
def gkey(s):
    s = re.sub(r"[^а-яәіңғүұқөһa-z]", "", match_form(s))
    return re.sub(r"(.)\1+", r"\1", s)

# --- 1. full KazSAnDRA, deduplicated exactly as before ---
os.environ.setdefault("HF_TOKEN", getpass("HF token: ")); login(os.environ["HF_TOKEN"])
full = pd.read_csv(hf_hub_download("issai/kazsandra", filename="full/full.csv", repo_type="dataset"))
full["text_norm"] = full["text"].map(norm_text)
full = full.drop_duplicates("text_norm").reset_index(drop=True)

# --- 2. labeled pool from the two sheets ---
f = pd.read_csv("data/label_flagged_split.csv", encoding="utf-8-sig");          f["src"] = "flagged"
s = pd.read_csv("data/label_unflagged_sample_split.csv", encoding="utf-8-sig"); s["src"] = "sample"
lab = pd.concat([f, s], ignore_index=True)
for c in ["kk_prof", "ru_prof", "proposed"]: lab[c] = pd.to_numeric(lab[c], errors="coerce")
lab["gkey"] = lab["text"].map(gkey)
rev = lab.kk_prof.notna() | lab.ru_prof.notna()
auto = (lab.src == "flagged") & lab.proposed.notna()
lab["y"] = np.select([rev, auto, lab.src == "sample"],
                     [((lab.kk_prof == 1) | (lab.ru_prof == 1)).astype(int), lab.proposed, 0], default=np.nan)
lab["label_source"] = np.select([rev, auto, lab.src == "sample"],
                                ["manual", "auto_lexicon", "assumed_negative"], default="excluded_unlabeled")
print(lab.label_source.value_counts())
allpool = lab.copy()
excl_keys = set(lab.loc[lab.y.isna(), "gkey"]) - {""}
lab = lab[lab.y.notna()].copy()

# --- 3. extra unflagged negatives, spread across domains ---
rest = full[~full.custom_id.isin(allpool.custom_id)].copy()       # every flagged review is in the sheets
rest["gkey"] = rest["text"].map(gkey)
rest = rest[~rest.gkey.isin(excl_keys)]
quota = {"market": 6000, "appstore": 8000, "mapping": 1500, "bookstore": 500}
rest = pd.concat([rest[rest.domain == d].sample(min(n, (rest.domain == d).sum()), random_state=42)
                  for d, n in quota.items()])
def hsplit(r):
    k = r.gkey if r.gkey else r.custom_id
    h = int(hashlib.md5(str(k).encode()).hexdigest(), 16) % 100
    return "train" if h < 70 else "val" if h < 85 else "test"
twin = allpool.drop_duplicates("gkey").set_index("gkey")["split"]; twin = twin[twin.index != ""]
rest["split"] = rest.gkey.map(twin)
m = rest.split.isna(); rest.loc[m, "split"] = rest[m].apply(hsplit, axis=1)
rest["y"] = 0; rest["label_source"] = "assumed_negative"

# --- 4. assemble, save ---
cols = ["custom_id", "domain", "label", "text", "y", "label_source", "split"]
data = pd.concat([lab[cols], rest[cols]], ignore_index=True).rename(columns={"label": "rating"})
data["y"] = data["y"].astype(int)
print(pd.crosstab(data.split, data.y, margins=True))
print(pd.crosstab([data.split, data.domain], data.y))
print(data[data.y == 1].label_source.value_counts())
data.to_csv("data/dataset_local.csv", index=False, encoding="utf-8-sig")              # has review text: keep OUT of git
data[["custom_id", "domain", "y", "label_source", "split"]].to_csv("data/labels_public.csv", index=False)
gi = Path(".gitignore"); t = gi.read_text() if gi.exists() else ""
if "data/dataset_local.csv" not in t:
    gi.write_text(t + "\ndata/dataset_local.csv\ndata/label_*.csv\nresults/*_local.csv\n")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


label_source
auto_lexicon          1130
excluded_unlabeled     946
assumed_negative       650
Name: count, dtype: int64
y          0     1    All
split                    
test    2462   169   2631
train  11608   788  12396
val     2580   173   2753
All    16650  1130  17780
y                   0    1
split domain              
test  appstore   1278  169
      bookstore    76    0
      mapping     224    0
      market      884    0
train appstore   5760  783
      bookstore   385    4
      mapping    1074    1
      market     4389    0
val   appstore   1262  173
      bookstore    82    0
      mapping     259    0
      market      977    0
label_source
auto_lexicon    1130
Name: count, dtype: int64


In [28]:
import json, re, unicodedata, joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
SEED = 42
Path("src").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True); Path("results").mkdir(exist_ok=True)
Path("src/kkprof.py").write_text(r"""
import re, unicodedata, joblib

def prep(s):
    s = unicodedata.normalize("NFKC", str(s)).lower().replace("ё", "е")
    s = re.sub(r"https?://\S+|\S+@\S+", " ", s)      # links / emails
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)             # 'ааааа' -> 'аа'
    return re.sub(r"\s+", " ", s).strip()

def load(path="models/tfidf_lr.joblib"):
    return joblib.load(path)

def predict(text, bundle=None):
    b = bundle or load()
    p = float(b["clf"].predict_proba(b["vec"].transform([prep(text)]))[0, 1])
    return {"label": int(p >= b["thr"]), "prob": round(p, 4), "threshold": round(b["thr"], 4)}
""", encoding="utf-8")
from src.kkprof import prep
data = pd.read_csv("data/dataset_local.csv", encoding="utf-8-sig")
tr, va, te = [data[data.split == k].reset_index(drop=True) for k in ("train", "val", "test")]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)
Xtr = vec.fit_transform(tr.text.map(prep)); Xva = vec.transform(va.text.map(prep))   # vocab fit on train only
best = None
for C in [0.3, 1, 3, 10, 30]:
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEED).fit(Xtr, tr.y)
    p = clf.predict_proba(Xva)[:, 1]
    prec, rec, thr = precision_recall_curve(va.y, p)
    f1 = 2*prec*rec/(prec+rec+1e-12); i = int(np.nanargmax(f1[:-1]))
    print(f"C={C}: val F1={f1[i]:.3f} thr={thr[i]:.3f} P={prec[i]:.3f} R={rec[i]:.3f}")
    if best is None or f1[i] > best[0]: best = (f1[i], C, float(thr[i]), clf)
f1v, C, thr, clf = best
joblib.dump({"vec": vec, "clf": clf, "thr": thr}, "models/tfidf_lr.joblib")
json.dump({"model": "char_wb TF-IDF (2-5) + LogisticRegression(class_weight=balanced)", "C": C, "threshold": thr,
           "seed": SEED, "val_f1": float(f1v), "max_features": 300000, "split_seed": 42,
           "split": "70/15/15 by near-duplicate group"}, open("models/config.json", "w"), indent=2)

C=0.3: val F1=0.947 thr=0.584 P=0.964 R=0.931
C=1: val F1=0.962 thr=0.562 P=0.982 R=0.942
C=3: val F1=0.968 thr=0.521 P=0.976 R=0.960
C=10: val F1=0.966 thr=0.367 P=0.955 R=0.977
C=30: val F1=0.969 thr=0.355 P=0.955 R=0.983


In [30]:
# ===== Cell F: FINAL TEST EVALUATION (run ONCE, after the model/threshold are frozen) =====
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
                             average_precision_score, roc_auc_score)

sys.path.insert(0, ".")
from src.kkprof import prep, load

FORCE = False                       # set True only if you deliberately re-evaluate (e.g. after a bug fix)
out = Path("results"); out.mkdir(exist_ok=True)
if (out / "metrics.json").exists() and not FORCE:
    raise SystemExit("results/metrics.json already exists: the test set was already evaluated. "
                     "Set FORCE=True only if you really mean to re-run.")

# --- 1. load frozen model + test split ---
b = load("models/tfidf_lr.joblib")
data = pd.read_csv("data/dataset_local.csv", encoding="utf-8-sig")
te = data[data.split == "test"].reset_index(drop=True)
te["prob"] = b["clf"].predict_proba(b["vec"].transform(te.text.map(prep)))[:, 1]
te["pred"] = (te.prob >= b["thr"]).astype(int)          # threshold chosen on VALIDATION only
y, p = te.y.values, te.pred.values

# --- 2. overall metrics (profanity class = 1) ---
tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
m = {
    "n_test": int(len(te)), "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()),
    "threshold": float(b["thr"]),
    "precision": float(precision_score(y, p, zero_division=0)),
    "recall": float(recall_score(y, p, zero_division=0)),
    "f1": float(f1_score(y, p, zero_division=0)),
    "pr_auc": float(average_precision_score(y, te.prob)),
    "roc_auc": float(roc_auc_score(y, te.prob)),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}

# --- 3. per-domain and per-label-source breakdown ---
def breakdown(g):
    t = int(((g.y == 1) & (g.pred == 1)).sum()); f_p = int(((g.y == 0) & (g.pred == 1)).sum())
    f_n = int(((g.y == 1) & (g.pred == 0)).sum())
    return {"n": int(len(g)), "pos": int(g.y.sum()), "pred_pos": int(g.pred.sum()),
            "tp": t, "fp": f_p, "fn": f_n,
            "precision": t / (t + f_p) if (t + f_p) else None,   # None = undefined
            "recall": t / (t + f_n) if (t + f_n) else None}
m["by_domain"] = {k: breakdown(g) for k, g in te.groupby("domain")}
m["positives_by_label_source"] = {k: breakdown(g) for k, g in te[te.y == 1].groupby("label_source")}
json.dump(m, open(out / "metrics.json", "w"), indent=2)

# --- 4. confusion matrix figure ---
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["pred 0", "pred 1"]); ax.set_yticks([0, 1], ["true 0", "true 1"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
ax.set_title(f"Test confusion matrix (n={len(te)}, thr={b['thr']:.3f})")
plt.tight_layout(); plt.savefig(out / "confusion_matrix.png", dpi=150); plt.close()

# --- 5. predictions (public: no text) and errors (local: with text, NOT committed) ---
te[["custom_id", "domain", "y", "pred", "prob", "label_source"]].round({"prob": 4}) \
  .to_csv(out / "test_predictions.csv", index=False)
err = te[te.y != te.pred].copy()
err["error"] = np.where(err.pred == 1, "FP", "FN")
err.sort_values(["error", "prob"], ascending=[True, False]) \
   [["custom_id", "domain", "error", "y", "pred", "prob", "label_source", "text"]] \
   .to_csv(out / "test_errors_local.csv", index=False, encoding="utf-8-sig")

# --- 6. print summary ---
print(f"n_test={m['n_test']}  pos={m['n_pos']}  neg={m['n_neg']}  thr={m['threshold']:.3f}")
print(f"P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  PR-AUC={m['pr_auc']:.3f}  ROC-AUC={m['roc_auc']:.3f}")
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print("\nPer domain:");        print(pd.DataFrame(m["by_domain"]).T)
print("\nPositives by label_source (recall = agreement with lexicon vs manual):")
print(pd.DataFrame(m["positives_by_label_source"]).T)
print(f"\nErrors: {len(err)} (FP={int((err.error=='FP').sum())}, FN={int((err.error=='FN').sum())}) -> results/test_errors_local.csv")
# peek (local only, do not paste into README):
for kind in ("FP", "FN"):
    print(f"\n--- top {kind} ---")
    for _, r in err[err.error == kind].sort_values("prob", ascending=(kind == "FN")).head(8).iterrows():
        print(f"[{r.domain} p={r.prob:.3f} {r.label_source}] {r.text[:150]!r}")

n_test=2631  pos=169  neg=2462  thr=0.355
P=0.954  R=0.982  F1=0.968  PR-AUC=0.994  ROC-AUC=1.000
TN=2454  FP=8  FN=3  TP=166

Per domain:
                n    pos  pred_pos     tp   fp   fn  precision    recall
appstore   1447.0  169.0     173.0  166.0  7.0  3.0   0.959538  0.982249
bookstore    76.0    0.0       1.0    0.0  1.0  0.0   0.000000       NaN
mapping     224.0    0.0       0.0    0.0  0.0  0.0        NaN       NaN
market      884.0    0.0       0.0    0.0  0.0  0.0        NaN       NaN

Positives by label_source (recall = agreement with lexicon vs manual):
                  n    pos  pred_pos     tp   fp   fn  precision    recall
auto_lexicon  169.0  169.0     166.0  166.0  0.0  3.0        1.0  0.982249

Errors: 11 (FP=8, FN=3) -> results/test_errors_local.csv

--- top FP ---
[appstore p=0.999 assumed_negative] 'Жасагандар қо ақбастАр издегенин шкпайд наху'
[appstore p=0.776 assumed_negative] 'Сен малсың мынаны тартпа далба. Н'
[bookstore p=0.612 assumed_negative] 'ешекбас

In [37]:
# ===== Cell E2: retrain with prep_v2 (fold Kazakh letters + masking). VALIDATION ONLY, no test data touched. =====
# Prereq: new src/kkprof.py in place (do NOT re-run the old cell E: it rewrites src/kkprof.py back to v1).
import json, sys, joblib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
sys.path.insert(0, ".")
from src.kkprof import prep_v2, predict

SEED = 42
data = pd.read_csv("data/dataset_local.csv", encoding="utf-8-sig")
tr, va = [data[data.split == k].reset_index(drop=True) for k in ("train", "val")]   # test is NOT loaded

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)
Xtr = vec.fit_transform(tr.text.map(prep_v2)); Xva = vec.transform(va.text.map(prep_v2))   # fit on train only
best = None
for C in [0.3, 1, 3, 10, 30]:                                    # same grid as v1
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEED).fit(Xtr, tr.y)
    p = clf.predict_proba(Xva)[:, 1]
    prec, rec, thr = precision_recall_curve(va.y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12); i = int(np.nanargmax(f1[:-1]))
    print(f"C={C}: val F1={f1[i]:.3f} thr={thr[i]:.3f} P={prec[i]:.3f} R={rec[i]:.3f}")
    if best is None or f1[i] > best[0]: best = (f1[i], C, float(thr[i]), clf)
f1v, C, thr, clf = best

v1_val = json.load(open("models/config.json"))["val_f1"]
print(f"\nvalidation F1: v1={v1_val:.4f}  v2={f1v:.4f}  (decide by validation only)")

joblib.dump({"vec": vec, "clf": clf, "thr": thr, "prep": "v2"}, "models/tfidf_lr_v2.joblib")
json.dump({"model": "char_wb TF-IDF (2-5) + LogisticRegression(class_weight=balanced)", "prep": "v2",
           "C": C, "threshold": thr, "seed": SEED, "val_f1": float(f1v), "max_features": 300000,
           "split_seed": 42, "split": "70/15/15 by near-duplicate group"},
          open("models/config_v2.json", "w"), indent=2)

# sanity: the spellings that v1 missed (synthetic probes, not from the test set)
for t in ["Котакбас", "қ0тақбас", "ешекбас бл*ть", "Маған қатты ұнады, жақсы өнім"]:
    print(t, "->", predict(t))

C=0.3: val F1=0.938 thr=0.627 P=0.952 R=0.925
C=1: val F1=0.948 thr=0.650 P=0.953 R=0.942
C=3: val F1=0.948 thr=0.721 P=0.953 R=0.942
C=10: val F1=0.952 thr=0.446 P=0.929 R=0.977
C=30: val F1=0.958 thr=0.382 P=0.929 R=0.988

validation F1: v1=0.9687  v2=0.9580  (decide by validation only)
Котакбас -> {'label': 1, 'prob': 1.0, 'threshold': 0.382}
қ0тақбас -> {'label': 1, 'prob': 1.0, 'threshold': 0.382}
ешекбас бл*ть -> {'label': 1, 'prob': 0.9426, 'threshold': 0.382}
Маған қатты ұнады, жақсы өнім -> {'label': 0, 'prob': 0.0, 'threshold': 0.382}


In [36]:
import importlib, src.kkprof; importlib.reload(src.kkprof)

<module 'src.kkprof' from 'C:\\Users\\Artyom\\PycharmProjects\\JupyterProject1\\src\\kkprof.py'>

In [38]:
import sklearn, pandas, numpy, scipy, joblib, huggingface_hub, matplotlib
for m in (sklearn, pandas, numpy, scipy, joblib, huggingface_hub, matplotlib):
    print(f"{m.__name__.replace('sklearn','scikit-learn')}=={m.__version__}")

scikit-learn==1.9.1
pandas==3.1.1
numpy==2.5.5
scipy==1.18.1
joblib==1.6.0
huggingface_hub==2.4.0
matplotlib==3.11.2


In [40]:
# ===== Cell E3: retrain v3 = masking handling + training on BOTH original and Kazakh-folded copies. =====
# VALIDATION ONLY (test is not loaded). Prereq: new src/kkprof.py in place; restart the kernel / reload the module.
import json, sys, importlib, joblib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
sys.path.insert(0, ".")
import src.kkprof as kk; importlib.reload(kk)
from src.kkprof import prep_v3, fold, predict

SEED = 42
data = pd.read_csv("data/dataset_local.csv", encoding="utf-8-sig")
tr, va = [data[data.split == k].reset_index(drop=True) for k in ("train", "val")]

# augmentation on TRAIN only: every training comment appears in its original spelling and its folded spelling
t0 = tr.text.map(prep_v3)
aug = pd.DataFrame({"t": pd.concat([t0, t0.map(fold)], ignore_index=True),
                    "y": pd.concat([tr.y, tr.y], ignore_index=True)}).drop_duplicates("t").reset_index(drop=True)
print(f"train rows {len(tr)} -> {len(aug)} after adding folded copies (positives {int(aug.y.sum())})")

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)
Xtr = vec.fit_transform(aug.t)                                   # fit on augmented train only
Xva = vec.transform(va.text.map(prep_v3))                        # validation stays un-augmented
best = None
for C in [0.3, 1, 3, 10, 30, 100]:                               # grid extended: v2 picked the edge (30)
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=SEED).fit(Xtr, aug.y)
    p = clf.predict_proba(Xva)[:, 1]
    prec, rec, thr = precision_recall_curve(va.y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12); i = int(np.nanargmax(f1[:-1]))
    print(f"C={C}: val F1={f1[i]:.3f} thr={thr[i]:.3f} P={prec[i]:.3f} R={rec[i]:.3f}")
    if best is None or f1[i] > best[0]: best = (f1[i], C, float(thr[i]), clf)
f1v, C, thr, clf = best

v1_val = json.load(open("models/config.json"))["val_f1"]
print(f"\nvalidation F1: v1={v1_val:.4f}  v3={f1v:.4f}   rule: adopt v3 only if v3 >= v1")

joblib.dump({"vec": vec, "clf": clf, "thr": thr, "prep": "v3"}, "models/tfidf_lr_v3.joblib")
json.dump({"model": "char_wb TF-IDF (2-5) + LogisticRegression(class_weight=balanced)", "prep": "v3",
           "augmentation": "train only: original + Kazakh-letter-folded copy of every comment",
           "C": C, "threshold": thr, "seed": SEED, "val_f1": float(f1v), "max_features": 300000,
           "split_seed": 42, "split": "70/15/15 by near-duplicate group"},
          open("models/config_v3.json", "w"), indent=2)

b3 = joblib.load("models/tfidf_lr_v3.joblib")
for t in ["Котакбас", "қотақбас", "қ0тақбас", "ешекбас бл*ть", "Маған қатты ұнады, жақсы өнім"]:   # synthetic probes
    print(t, "->", predict(t, b3))

train rows 12396 -> 22984 after adding folded copies (positives 1441)
C=0.3: val F1=0.942 thr=0.615 P=0.947 R=0.936
C=1: val F1=0.942 thr=0.687 P=0.947 R=0.936
C=3: val F1=0.951 thr=0.531 P=0.943 R=0.960
C=10: val F1=0.950 thr=0.775 P=0.964 R=0.936
C=30: val F1=0.955 thr=0.425 P=0.939 R=0.971
C=100: val F1=0.955 thr=0.256 P=0.924 R=0.988

validation F1: v1=0.9687  v3=0.9553   rule: adopt v3 only if v3 >= v1
Котакбас -> {'label': 1, 'prob': 1.0, 'threshold': 0.2563}
қотақбас -> {'label': 1, 'prob': 1.0, 'threshold': 0.2563}
қ0тақбас -> {'label': 1, 'prob': 1.0, 'threshold': 0.2563}
ешекбас бл*ть -> {'label': 1, 'prob': 0.9956, 'threshold': 0.2563}
Маған қатты ұнады, жақсы өнім -> {'label': 0, 'prob': 0.0, 'threshold': 0.2563}


In [41]:
import sklearn, pandas, numpy, scipy, joblib, huggingface_hub, matplotlib
mods = [sklearn, pandas, numpy, scipy, joblib, huggingface_hub, matplotlib]
lines = [f"{m.__name__.replace('sklearn', 'scikit-learn')}=={m.__version__}" for m in mods]
open("requirements.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

scikit-learn==1.9.1
pandas==3.1.1
numpy==2.5.5
scipy==1.18.1
joblib==1.6.0
huggingface_hub==2.4.0
matplotlib==3.11.2
